# Session 10: Fourier analysis

The scientist who sets up a routine ¹H method on a shared NMR spectrometer chooses how long each scan listens, the acquisition time, and too short a time merges a multiplet into one line. This notebook turns a real free induction decay into a spectrum with `numpy.fft`, then asks how short that decay can be cut and still show ethanol's CH₃ triplet as three lines. Section 1 builds the spectrum of the full record, checks it against the chemistry, and shows where a line outside the spectral window lands. Section 2 finds the shortest acquisition time that resolves the triplet in your own record. Section 3 compares two ways to cut a decay, a hard cut and a taper, and asks which one's resolution is real. Section 4 is the judgment cell.

The FID is real: Biological Magnetic Resonance Data Bank (BMRB), entry bmse000297, ethanol, 1D ¹H spectrum, experiment set01 ([doi:10.13018/BMSE000297](https://doi.org/10.13018/BMSE000297)), files `fid` and `acqus`, disseminated by BMRB in the public domain, accessed 2026-10-10. We acknowledge BMRB as the source, as its terms of use request. `data/README.md` describes the two files and `data/records.csv`, which gives each student's record.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 10 assignment on Canvas by 1:30 pm on Thursday, November 5. Submissions are individual. Every record is rebuilt from the FID in `data/` and its seed in `data/records.csv`, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 1, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas and Matplotlib. Nothing new is imported: the Fourier transform is `np.fft`, part of NumPy, and this session uses four of its functions. `np.fft.fft` turns a series of points into its frequency components, `np.fft.ifft` turns them back, `np.fft.fftfreq` gives the frequency of each component, and `np.fft.fftshift` puts the negative frequencies before the positive ones, so the axis runs low to high. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parameters

The two data files and the table of records; the processing every spectrum shares (line broadening, zero-fill); the regions of the spectrum used to reference, phase, find the triplet and measure the noise; the acquisition times sections 2 and 3 try and the test for "resolved"; then your record, set in section 2. Leave `RECORD` as `CHANGE-ME` until section 2.

In [ ]:
FID_PATH = Path("data/fid")                  # BMRB bmse000297: the raw FID, big-endian 32-bit integers
ACQUS_PATH = Path("data/acqus")              # the spectrometer's acquisition parameters for that FID
RECORDS_CSV = Path("data/records.csv")       # each student's record name and its seed
RNG_SEED = 427                               # make_records.py tried seeds from here; your record's own seed is in records.csv
LB_HZ = 0.3                                  # exponential line broadening, Hz, in every spectrum today
ZERO_FILL = 65536                            # points in every spectrum, so every spectrum shares one axis
DSS_WINDOW_PPM = 0.3                         # DSS is the tallest line within this distance of 0 ppm
CH3_REGION_PPM = (1.10, 1.30)                # the CH3 triplet: where the phase is set and its lines are found
NOISE_REGION_PPM = (7.0, 9.0)                # no lines here in the full spectrum: the noise for S/N
EMPTY_ABOVE_PPM = 5.0                        # the full spectrum has no line above this
AQ_LIST_S = np.round(np.arange(0.10, 0.501, 0.02), 2)   # acquisition times tried in sections 2 and 3, s
VALLEY_MAX = 0.8                             # resolved: both valleys at most 80% of the outer line beside them
LOBE_WINDOW_HZ = 15                          # negative lobes are looked for within this distance of the centre line
RECORD = "CHANGE-ME"                         # your assigned record, for example "record_07", section 2

## 1. Explore: from an FID to a spectrum you can check

An NMR spectrometer records the free induction decay, or FID: the signal the nuclei induce in the coil after a pulse, sampled at a fixed interval. Dataset A is one ¹H FID of 100 mM ethanol in D₂O with DSS, recorded on a Bruker 400 MHz spectrometer. The first cell reads `acqus`, the spectrometer's parameter file, with `read_acqus`, a helper given complete. It reads `fid` as big-endian 32-bit integers, pairs them into complex points, and checks the count and the acquisition time against `acqus`.

In [ ]:
def read_acqus(path):
    """The ##$ parameters of a Bruker acqus file, as a dict of numbers and strings."""
    params = {}
    for line in Path(path).read_text(encoding="ascii").splitlines():
        if line.startswith("##$") and "= " in line:
            key, value = line[3:].split("= ", 1)
            try:
                params[key] = float(value)
            except ValueError:
                params[key] = value
    return params


acqus = read_acqus(ACQUS_PATH)
sw_hz = acqus["SW_h"]                   # spectral width, Hz: points are 1 / sw_hz apart
sfo1_mhz = acqus["SFO1"]                # the spectrometer frequency for 1H, MHz
o1_hz = acqus["O1"]                     # the carrier, Hz above the reference frequency
raw = np.fromfile(FID_PATH, dtype=">i4")   # TD values, real and imaginary interleaved
fid_raw = raw[0::2] + 1j * raw[1::2]       # complex points
n_points = fid_raw.size
dwell_s = 1 / sw_hz                     # s between points
aq_s = n_points * dwell_s               # acquisition time, s
print(f"TD {acqus['TD']:.0f} in acqus; {raw.size} values read, {n_points} complex points")
print(f"SW_h {sw_hz:.2f} Hz, dwell {dwell_s * 1e6:.1f} us, AQ {aq_s:.3f} s")
print(f"SFO1 {sfo1_mhz:.6f} MHz, O1 {o1_hz:.1f} Hz, the carrier at {o1_hz / sfo1_mhz:.3f} ppm")
print(f"largest |FID| at point {np.argmax(np.abs(fid_raw))}, not 0")

The FID peaks at point 82, not 0, because the spectrometer's digital filter delays the signal by 72.138 points at these settings and its response takes about 10 more to rise; `remove_filter_delay`, given complete, takes the delay out. The next cell multiplies the FID by a decaying exponential (0.3 Hz of line broadening), zero-fills it to 65,536 points and transforms it. The axis is in Hz from the carrier, from `np.fft.fftfreq` and the dwell time, then in ppm as (Hz + O1) / SFO1. Two given helpers set DSS to 0 ppm and choose the one phase that makes the CH₃ lines positive.

In [ ]:
def remove_filter_delay(fid, decim, dspfvs):
    """Undo the Bruker digital filter's delay, a fraction of a point included, as a phase ramp."""
    delay = {(32, 12): 72.138}[(int(decim), int(dspfvs))]   # points, Bruker's table for these settings
    k = np.fft.fftfreq(fid.size) * fid.size                  # signed: negative frequencies get a negative ramp
    return np.fft.ifft(np.fft.fft(fid) * np.exp(2j * np.pi * delay * k / fid.size))


def reference_to_dss(ppm, spec):
    """Shift the ppm axis so that the tallest line near 0 ppm, DSS, is at 0."""
    near = np.abs(ppm) < DSS_WINDOW_PPM
    return ppm - ppm[near][np.argmax(np.abs(spec[near]))]


def phase_zero_order(spec, ppm):
    """The phase, in radians, that makes the real part of the CH3 region as large as it can be."""
    region = (ppm > CH3_REGION_PPM[0]) & (ppm < CH3_REGION_PPM[1])
    phases = np.linspace(-np.pi, np.pi, 720, endpoint=False)
    return phases[np.argmax([np.sum(np.real(spec[region] * np.exp(1j * p))) for p in phases])]


def spectrum(x, n_keep, taper=False):
    """The spectrum of the first n_keep points of x: line broadening, an optional cos^2 taper, zero-fill, fft."""
    window = np.exp(-np.pi * LB_HZ * np.arange(n_keep) * dwell_s)
    if taper:
        window = window * np.cos(0.5 * np.pi * np.arange(n_keep) / n_keep) ** 2
    y = x[:n_keep] * window
    y[0] = y[0] / 2                     # the first point is counted once, not twice
    return np.fft.fftshift(np.fft.fft(y, ZERO_FILL))


fid = remove_filter_delay(fid_raw, acqus["DECIM"], acqus["DSPFVS"])
spec_a = spectrum(fid, n_points)
hz = np.fft.fftshift(np.fft.fftfreq(ZERO_FILL, d=dwell_s))   # Hz from the carrier, lowest first
ppm_unreferenced = (hz + o1_hz) / sfo1_mhz
ppm = reference_to_dss(ppm_unreferenced, spec_a)
dss_shift_ppm = ppm[0] - ppm_unreferenced[0]
phase = phase_zero_order(spec_a, ppm)                        # radians, used for every spectrum today
real_a = np.real(spec_a * np.exp(1j * phase))

fig, (ax_all, ax_ch3) = plt.subplots(1, 2, figsize=(10, 4))
ax_all.plot(ppm, real_a / 1e6, color="black", linewidth=0.8)
ax_all.set_xlim(5.2, -0.3)
ax_all.set_xlabel("chemical shift (ppm)")
ax_all.set_ylabel("intensity (millions)")
ax_all.set_title("Dataset A: ethanol in D2O, 3.4 s FID")
in_ch3 = (ppm > 1.12) & (ppm < 1.22)
ax_ch3.plot(ppm[in_ch3], real_a[in_ch3] / 1e6, color="black")
ax_ch3.set_xlim(1.22, 1.12)
ax_ch3.set_xlabel("chemical shift (ppm)")
ax_ch3.set_ylabel("intensity (millions)")
ax_ch3.set_title("the CH3 triplet")
fig.tight_layout()
plt.show()

The full spectrum, from 5.2 to −0.3 ppm, has HDO at 4.807 ppm as its tallest line, 3.0 times the CH₃ centre line, the CH₂ quartet at 3.615 to 3.669 ppm, the CH₃ triplet at 1.153, 1.171 and 1.188 ppm, and DSS at 0 ppm; the right panel shows the triplet's three lines, heights 0.48 : 1 : 0.52, each falling to near the baseline before the next.

To check the axis against the chemistry, the next cell finds the three tallest local maxima in the CH₃ region, the triplet's lines, and turns their spacings from ppm into Hz with SFO1: ppm times MHz is Hz. Ethanol's CH₃ to CH₂ coupling, J, is near 7 Hz. It measures the centre line's width at half height, walking out from its top until the spectrum falls below half, and prints it beside 1/AQ, the finest spacing this acquisition can separate. It also defines `valley_ratio`, which sections 2 and 3 use: the lowest point between each outer line and the centre, as a fraction of that outer line.

In [ ]:
region = np.flatnonzero((ppm > CH3_REGION_PPM[0]) & (ppm < CH3_REGION_PPM[1]))
y = real_a[region]
tops = np.flatnonzero((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:])) + 1   # local maxima in the region
lines = np.sort(region[tops[np.argsort(y[tops])[-3:]]])               # the three tallest: the triplet
j_hz = np.diff(ppm[lines]) * sfo1_mhz                                 # line spacings, ppm times MHz is Hz
hz_per_point = sw_hz / ZERO_FILL                                      # Hz between points of the spectrum

top = lines[1]
half = real_a[top] / 2
left = top - np.argmax(real_a[top::-1] < half)        # first point below half, walking left
right = top + np.argmax(real_a[top:] < half)          # and walking right
width_a_hz = (right - left) * hz_per_point

noise_region = (ppm > NOISE_REGION_PPM[0]) & (ppm < NOISE_REGION_PPM[1])
noise_a = real_a[noise_region].std()


def valley_ratio(real_spectrum, lines):
    """The larger of the two valleys between the triplet's lines, each as a fraction of the outer line beside it."""
    heights = np.array([real_spectrum[i - 3:i + 4].max() for i in lines])   # within 0.2 Hz of each line
    left = real_spectrum[lines[0]:lines[1] + 1].min() / heights[0]
    right = real_spectrum[lines[1]:lines[2] + 1].min() / heights[2]
    return max(left, right) if heights[1] == heights.max() else np.inf     # the centre line must be the tallest


print("triplet at " + ", ".join(f"{p:.3f}" for p in ppm[lines]) + " ppm; heights "
      + " : ".join(f"{h:.2f}" for h in real_a[lines] / real_a[top]))
print(f"J from the spacings: {j_hz[0]:.2f} and {j_hz[1]:.2f} Hz")
print(f"centre line {width_a_hz:.2f} Hz wide at half height; 1/AQ = {1 / aq_s:.2f} Hz")
print(f"S/N of the outer line {real_a[lines[0]] / noise_a:.0f}; larger valley {valley_ratio(real_a, lines):.2f}")

Points 1/SW_h apart can only tell apart frequencies within ±SW_h/2 of the carrier, the Nyquist limit: here ±2,403.8 Hz, a window 12.02 ppm wide. A line outside the window does not vanish. It appears folded back in, shifted by one window width. The next cell keeps every second point, which doubles the dwell and halves the window, predicts where the CH₃ centre line at 1.171 ppm should land, finds the tallest line above 5 ppm, where the full spectrum has none, and plots both spectra.

In [ ]:
dwell_half_s = 2 * dwell_s                             # every second point: the dwell doubles
window_half_hz = 1 / dwell_half_s                      # and the window halves, Hz
y = fid[::2] * np.exp(-np.pi * LB_HZ * np.arange(n_points // 2) * dwell_half_s)
y[0] = y[0] / 2
real_half = np.real(np.fft.fftshift(np.fft.fft(y, ZERO_FILL)) * np.exp(1j * phase))
hz_half = np.fft.fftshift(np.fft.fftfreq(ZERO_FILL, d=dwell_half_s))
ppm_half = (hz_half + o1_hz) / sfo1_mhz + dss_shift_ppm  # the same DSS reference as the full spectrum
predicted_ppm = ppm[lines[1]] + window_half_hz / sfo1_mhz
above = ppm_half > EMPTY_ABOVE_PPM
found_ppm = ppm_half[above][np.argmax(real_half[above])]
print(f"full window {ppm.min():.2f} to {ppm.max():.2f} ppm; every second point {ppm_half.min():.2f} to {ppm_half.max():.2f} ppm")
print(f"CH3 centre line {ppm[lines[1]]:.3f} ppm + one window {window_half_hz / sfo1_mhz:.3f} ppm = {predicted_ppm:.3f} ppm predicted")
print(f"tallest line above {EMPTY_ABOVE_PPM:g} ppm with every second point: {found_ppm:.3f} ppm")

fig, ax = plt.subplots()
ax.plot(ppm, real_a / real_a[lines[1]] + 1.5, color="gray", linewidth=0.8, label="every point (offset by 1.5)")
ax.plot(ppm_half, real_half / real_half[above].max(), color="black", linestyle="-", linewidth=0.8, label="every second point")
ax.set_xlim(8.0, -0.3)
ax.set_ylim(-0.3, 3.0)
ax.set_xlabel("chemical shift (ppm)")
ax.set_ylabel("intensity, scaled to the CH3 centre line")
ax.set_title("Dataset A: the window halved")
ax.legend()
fig.tight_layout()
plt.show()

With every point (gray, offset by 1.5) the lines sit at 4.81 ppm (HDO), 3.6 ppm (CH₂) and 1.17 ppm (CH₃); with every second point (black) the window runs from 1.80 to 7.81 ppm, HDO and the quartet stay where they were, the triplet is gone from 1.17 ppm, and a triplet stands at 7.178 ppm.

**Explain.** In two sentences: a colleague looks at the every-second-point spectrum, sees a triplet near 7.18 ppm and calls it an aromatic impurity. What in your output shows it is the CH₃ triplet, with the numbers?

*Your answer:*

**Ask the assistant** for J in your own words. Save the notebook first. Add a markdown cell under this one (**+ Markdown**) holding the lines below, and write your prompt on the first before you send it. Say where the spectrum is: `activity.ipynb` in this session's folder. Read the answer against this section's J: its number, unit and decimals, any ± and what sets it, and the conversion from ppm to Hz. Delete any cell the assistant adds (**Delete Cell**). Section 2 fills the other lines.

```
**Prompt record**

- **My prompt:**
- **The line I wrote in CLAUDE.local.md, and what in the first answer it changes:**
- **What the second answer showed, against section 1's J:**
```

## 2. Apply: find the shortest acquisition time for your record

Your record is Dataset A's FID with two changes that differ by student: an extra exponential decay of 0.2 to 2 Hz, the broadening a poorer shim gives, and added noise. So your lines are wider and your noise higher than Dataset A's, by amounts you measure. The question for your record: how short can the FID be cut, by keeping only its first points, and still show the triplet as three lines?

**Keep a CLAUDE.md**, the second half of today's AI practice, before TASK 1. In VS Code's Explorer, right-click the top folder of your clone of the course repository, the one that holds `CLAUDE.md` and `README.md`, choose **New File...**, and name it `CLAUDE.local.md`; not in this session's folder. If `CLAUDE.md` is not in the Explorer, open the clone with **File**, then **Open Folder...** first.

Write one line in it, your own: a convention you want every answer to follow, chosen from what the first answer did not do the way you want. It can fix a unit, a number of decimals, an uncertainty and what sets its size, or a step to show, such as the conversion from ppm to Hz. Write it so that the next answer either visibly follows it or does not; notes section 6 says what makes a line work. Save the file, and copy the line onto the second line of your prompt record, with what in the first answer it changes.

In the Claude Code prompt box, type `/` and choose **Clear conversation**, so the assistant starts again and reads the new file. Type `/memory` to see that it is listed. Then send the same prompt as at the end of section 1. `git status` will not list `CLAUDE.local.md`, because the repository ignores it, and a fresh clone will not have it, so keep a copy.

Read the second answer beside the first, against section 1's printout. Did the second answer follow each part of your line, and is its J still the one section 1 printed? Write that on the third line of your record. A line that did not visibly change the answer is a finding too: say so, and what you would write instead. Your prompt record is one candidate for part (b) of the judgment cell.

**[TASK 1]** Set `RECORD` in the Parameters cell to your assigned record, run the Parameters cell, then run the next cell. It looks up your record's seed in `data/records.csv` and builds your FID from Dataset A's with `make_record`, a helper given complete. Until `RECORD` is set, it stops with `AssertionError: no record named 'CHANGE-ME' in data/records.csv; set RECORD in Parameters`.

In [ ]:
def make_record(fid, seed, dwell_s):
    """Dataset A's FID with a seeded extra decay of 0 to 2 Hz and seeded complex noise: one student's record."""
    rng = np.random.default_rng(seed)
    extra_lb_hz = rng.uniform(0.0, 2.0)          # a poorer shim, Hz
    sigma = 10 ** rng.uniform(0.0, 1.2)           # noise per point, in the FID's units
    t = np.arange(fid.size) * dwell_s
    noise = rng.normal(0, sigma, fid.size) + 1j * rng.normal(0, sigma, fid.size)
    return fid * np.exp(-np.pi * extra_lb_hz * t) + noise


records = pd.read_csv(RECORDS_CSV)
my_seed = records.loc[records["record"] == RECORD, "seed"]
assert len(my_seed) == 1, f"no record named {RECORD!r} in {RECORDS_CSV.as_posix()}; set RECORD in Parameters"
my_fid = make_record(fid, int(my_seed.iloc[0]), dwell_s)
real_b = np.real(spectrum(my_fid, n_points) * np.exp(1j * phase))
noise_b = real_b[noise_region].std()
print(f"{RECORD}: seed {int(my_seed.iloc[0])}, {my_fid.size} complex points, AQ {aq_s:.3f} s")
print(f"S/N of the outer line {real_b[lines[0]] / noise_b:.0f}, against {real_a[lines[0]] / noise_a:.0f} for Dataset A")

**[TASK 2]** Measure the centre line's width at half height on your full record, in Hz, in a code cell you add under this one and write yourself (hover just below this cell and click **+ Code**). Use the method section 1 used for Dataset A: walk out from the top of the centre line on each side until the spectrum falls below half its height. Your cell sets `width_b_hz`. `real_b` is your spectrum, `lines[1]` is the index of its centre line, and `hz_per_point` is the spacing of the spectrum's points in Hz. The cell after it prints your width beside Dataset A's and stops with `NameError: name 'width_b_hz' is not defined` until your cell sets it.

In [ ]:
print(f"{RECORD}: centre line {width_b_hz:.2f} Hz wide at half height, against {width_a_hz:.2f} Hz for Dataset A")   # set by the code cell you added above

**[TASK 3]** Cut your FID to each acquisition time in `AQ_LIST_S` and test whether the triplet is resolved. For each time, the cell keeps the first `n_keep` complex points, transforms them as section 1 did, and prints the time those points span, `n_keep / sw_hz`, beside 1/AQ and the larger valley ratio. Resolved means that ratio is at most 0.8. Check the printed `aq_s` column against `aq_target_s`. Until `n_keep` is filled, the cell stops with `TypeError: int() argument must be a string, a bytes-like object or a real number, not 'ellipsis'`.

In [ ]:
rows = []
for aq_target_s in AQ_LIST_S:
    n_keep = int(...)   # [TASK 3] the number of complex points that spans aq_target_s seconds
    real_cut = np.real(spectrum(my_fid, n_keep) * np.exp(1j * phase))
    ratio = valley_ratio(real_cut, lines)
    rows.append({"aq_target_s": aq_target_s, "n_keep": n_keep, "aq_s": n_keep / sw_hz,
                 "one_over_aq_hz": sw_hz / n_keep, "valley_ratio": ratio, "resolved": bool(ratio <= VALLEY_MAX)})
cuts = pd.DataFrame(rows)
print(cuts.round(3).to_string(index=False))

**[TASK 4]** Read your call from the table: the shortest acquisition time at which the triplet is resolved. The figure draws your spectrum at that time, one step shorter, and in full, each scaled to its own centre line; the test reads each valley against the outer line beside it, so a valley can look deep in the figure and still fail. Until it is filled, the cell stops with `TypeError: float() argument must be a string or a real number, not 'ellipsis'`.

In [ ]:
call_s = float(...)   # [TASK 4] the shortest aq_target_s in cuts at which resolved is True
step_before = cuts.loc[cuts["aq_target_s"] < call_s, "n_keep"].max()
n_call = int(cuts.loc[cuts["aq_target_s"] == call_s, "n_keep"].iloc[0])
print(f"{RECORD}: resolved from {call_s:.2f} s, {n_call} points, 1/AQ {sw_hz / n_call:.2f} Hz")

near = (ppm > 1.12) & (ppm < 1.22)
fig, ax = plt.subplots()
for n_keep, style, label in [(n_points, "-", f"full record, {aq_s:.2f} s"), (n_call, "--", f"{call_s:.2f} s, your call"),
                             (step_before, ":", f"{step_before / sw_hz:.2f} s, one step shorter")]:
    if n_keep > 0:
        cut = np.real(spectrum(my_fid, int(n_keep)) * np.exp(1j * phase))
        ax.plot(ppm[near], cut[near] / cut[lines[1]], color="black", linestyle=style, label=label)
ax.set_xlim(1.22, 1.12)
ax.set_xlabel("chemical shift (ppm)")
ax.set_ylabel("intensity, scaled to the centre line")
ax.set_title(f"{RECORD}: the CH3 triplet cut short")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** Before you compare: in two sentences, give your shortest acquisition time and your line width from TASK 2, and set 1/AQ at your call beside J. Then compare with a neighbor whose line width differs from yours: set your records, widths and calls side by side. Add one sentence on why two records whose widths differ give the same call, or nearly the same, close to 1/J = 0.14 s.

*Your answer:*

## 3. Analyze: choose between a hard cut and a taper

Cutting an FID before it has decayed multiplies it by a step: full height up to the last kept point, zero after. In the spectrum the step puts wiggles on every line, with negative lobes on either side, and a lobe can dig a valley between two lines that the lines alone would not make. A phased spectrum of real lines has no negative intensity. Treatment A is section 2's: the hard cut with 0.3 Hz of line broadening. Treatment B also multiplies the kept points by a cos² taper, which falls smoothly to zero at the last point, so there is no step, at the cost of wider lines. The next cell runs both at every acquisition time in your table. It prints each treatment's call, and at that call the larger valley, the deepest negative lobe within 15 Hz of the centre line as a fraction of it, the centre line's width, and the splitting the three tallest maxima show, in Hz, to set beside section 1's J. It then draws A at its call beside B at both calls.

In [ ]:
def width_at_half(real_spectrum, top):
    """Width at half height, in Hz, of the line whose top is at index top."""
    half = real_spectrum[top] / 2
    left = top - np.argmax(real_spectrum[top::-1] < half)
    right = top + np.argmax(real_spectrum[top:] < half)
    return (right - left) * hz_per_point


def apparent_j_hz(real_spectrum):
    """Mean spacing, in Hz, of the three tallest local maxima in the CH3 region: the splitting the spectrum shows."""
    y = real_spectrum[region]
    tops = np.flatnonzero((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:])) + 1
    three = np.sort(region[tops[np.argsort(y[tops])[-3:]]])
    return np.diff(hz[three]).mean() if three.size == 3 else np.nan


near_centre = np.abs(hz - hz[lines[1]]) < LOBE_WINDOW_HZ
rows = []
for cut in cuts.itertuples():
    for treatment, taper in [("A, hard cut", False), ("B, cos^2 taper", True)]:
        real_cut = np.real(spectrum(my_fid, cut.n_keep, taper) * np.exp(1j * phase))
        rows.append({"treatment": treatment, "aq_target_s": cut.aq_target_s, "valley_ratio": valley_ratio(real_cut, lines),
                     "negative_lobe": max(0.0, -real_cut[near_centre].min() / real_cut[lines[1]]),
                     "width_hz": width_at_half(real_cut, lines[1]), "apparent_j_hz": apparent_j_hz(real_cut)})
both = pd.DataFrame(rows)
at_call = both[both["valley_ratio"] <= VALLEY_MAX].groupby("treatment").first()
print(at_call.round(3).to_string())
call_a_s, call_b_s = at_call["aq_target_s"].iloc[0], at_call["aq_target_s"].iloc[1]
print(f"B's call is {call_b_s - call_a_s:.2f} s later than A's; 1/AQ {1 / call_a_s:.1f} Hz at A's call, {1 / call_b_s:.1f} Hz at B's, J {j_hz.mean():.2f} Hz")

fig, ax = plt.subplots()
for aq_cut, taper, style, label in [(call_a_s, False, "-", f"A, hard cut, {call_a_s:.2f} s"),
                                     (call_a_s, True, ":", f"B, taper, {call_a_s:.2f} s"),
                                     (call_b_s, True, "--", f"B, taper, {call_b_s:.2f} s")]:
    n_keep = int(cuts.loc[cuts["aq_target_s"] == aq_cut, "n_keep"].iloc[0])
    real_cut = np.real(spectrum(my_fid, n_keep, taper) * np.exp(1j * phase))
    ax.plot(ppm[near], real_cut[near] / real_cut[lines[1]], color="black", linestyle=style, label=label)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlim(1.22, 1.12)
ax.set_xlabel("chemical shift (ppm)")
ax.set_ylabel("intensity, scaled to the centre line")
ax.set_title(f"{RECORD}: a hard cut against a cos² taper")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** In two sentences, say which treatment's "resolved" you would trust for an assignment and why, citing the negative lobe and the splitting at A's call against section 1's J, and what B's call costs in acquisition time against A's. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the scientist setting up a routine ¹H method for this sample on this spectrometer, what acquisition time would you set? Cite your calls under treatments A and B, the negative lobe at A's call and your record's line width in Hz, and say what a wrong call would cost either way. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. Your prompt record from section 1 is the natural source; the reason has to cite a value from your output, such as section 1's J in Hz against the number the assistant reported, not a general rule. This session's AI practice, named in the notes, is *keep a CLAUDE.md*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"Matplotlib {matplotlib.__version__}")